In [3]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [4]:
load_dotenv(override=True)

True

In [5]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence."
request += "Answer only with the question, no explanation."
messages =  [{"role":"user", "content": request}]

In [6]:
messages

[{'role': 'user',
  'content': 'Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence.Answer only with the question, no explanation.'}]

In [7]:
#esta es la llamada de OpenAI
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API exists")
else:
    print("OpenAI not set")

OpenAI API exists


In [ ]:

###### Llamada a la api
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages
)
question = response.choices[0].message.content
print (question)

In [ ]:
messages = [{"role":"user", "content":question}]
competitors = []
answers = []

In [ ]:
model_name = "gpt-4.1-nano"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
#esta es la llamada de Anthropic (claude)
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
if anthropic_api_key:
    print(f"Anthropic API exists and begins {anthropic_api_key}")
else:
    print("anthropic_api_key not set")

In [ ]:
###### Llamada a la api
claude = Anthropic()
model_name = "claude-opus-4-6"
response = claude.messages.create(model=model_name, messages=messages,max_tokens=1000)
answer = response.content[0].text

display(Markdown(answer))


In [ ]:
competitors.append(model_name)
answers.append(answer)

In [ ]:
google_api_key = os.getenv('GOOGLE_API_KEY')
if google_api_key:
    print (f"google API exists and begins {google_api_key}")
else:
    print ("google_api_key does not exist")

In [ ]:
gemini = OpenAI(api_key=google_api_key,base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display (Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
#deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
#if deepseek_api_key:
#    print (f"deepseek_api_key API exists and begins {deepseek_api_key[:8]}")
#else:
#    print ("deepseek_api_key does not exist")

In [ ]:
print(competitors)
print(answers)

In [ ]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")

In [ ]:
together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
print (together)

In [8]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""

NameError: name 'competitors' is not defined

In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]

In [ ]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-4.1-nano",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)

In [ ]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")